In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats

def randu(seed, start, n_iter):
  np.random.seed(seed)
  res = np.zeros(n_iter, dtype=int)
  res[0] = start
  for i in range(1, n_iter):
    res[i] = (res[i-1] * 65539) % (2**31)
  return res/(2**31)


In [ ]:
my_pseudo_random_numbers = randu(0, 2, 20000)
my_idxs = np.argwhere((my_pseudo_random_numbers >= 0.50)*(my_pseudo_random_numbers <= 0.51))
plt.scatter(my_pseudo_random_numbers[my_idxs-1], my_pseudo_random_numbers[my_idxs])


In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure()

ax = fig.add_subplot(111, projection='3d')
ax.view_init(0, 57)
ax.scatter(my_pseudo_random_numbers[my_idxs-2], my_pseudo_random_numbers[my_idxs-1], my_pseudo_random_numbers[my_idxs])


In [ ]:
def laplace(seed, n_samples):
  """
  Sample from a Laplace distribution using two different methods:
    i) Difference between two independent exponential distributions
    ii) Draw a uniform, depending on its value (<= 0.5 or above), take the opposite of an exponential draw
      or the exponential draw.
  """
  np.random.seed(seed)
  u1 = np.random.uniform(size=n_samples)
  u2 = np.random.uniform(size=n_samples)
  exps1 = - np.log(u1)
  exps2 = -np.log(u2)
  signs = np.sign(-1+2*np.random.uniform(size=n_samples))
  return exps2-exps1, signs*exps1

In [ ]:
samples1, samples2 = laplace(0, 10000)
plt.hist(samples1, bins=100, density=True)
plt.hist(samples2, bins=100, density=True, alpha=0.5)

def laplace_pdf(x):
  return 0.5 * np.exp(-np.abs(x))


xs = np.linspace(-5, 5, 100)
ys = np.vectorize(laplace_pdf)(xs)
plt.plot(xs, ys)

In [ ]:
import jax
import jax.numpy as jnp

OP_key = jax.random.PRNGKey(0)
my_keys = jax.random.split(OP_key, 100000000)
def laplace(key):
  u1 = jax.random.uniform(key)
  exps1 = - jnp.log(u1)
  return exps1

jax.vmap(laplace)(my_keys)

In [ ]:
def rejection_sampling(seed, n_samples):
  np.random.seed(seed)
  laplace_draws, _ = laplace(seed, n_samples)
  uniform_draws = np.random.uniform(size=n_samples)
  M = np.sqrt(2*np.exp(1)/np.pi)
  accept_ratio = stats.norm.pdf(laplace_draws)/laplace_pdf(laplace_draws) * 1/M
  idxs = np.argwhere(uniform_draws<=accept_ratio)
  my_samples = laplace_draws[idxs]
  return my_samples

normal_samples = rejection_sampling(0, 10000)

In [ ]:
plt.hist(normal_samples, bins=100, density=True)